https://huggingface.co/bartowski/Qwen2.5-72B-Instruct-GGUF

https://github.com/IST-DASLab/gptq-gguf-toolkit/tree/main

https://huggingface.co/zed122/gptq-gguf-toolkit

https://github.com/ggml-org/llama.cpp/releases

In [ ]:
https://github.com/IST-DASLab/gptq-gguf-toolkit.git

In [ ]:
!git clone https://github.com/IST-DASLab/gptq-gguf-toolkit.git

In [ ]:
%cd gptq-gguf-toolkit

In [ ]:
!git pull

In [ ]:
!curl -LsSf https://llama.app/install.sh | sh

In [ ]:
%cd /content

In [ ]:
!wget https://github.com/ggml-org/llama.cpp/releases/download/b11721/llama-b11721-bin-ubuntu-x64.tar.gz

In [ ]:
!tar -xf /content/llama-b11721-bin-ubuntu-x64.tar.gz -C /content/

In [12]:
%cd /content/gptq-gguf-toolkit

/content/gptq-gguf-toolkit


In [ ]:
%cd quant/gguf
# تشغيل السكربت على النموذج الصغير وبصيغة واحدة فقط
!bash run_quant.sh /content/Qwen2.5-1.5B-Instruct-f16.gguf "Q4_K"

In [8]:
!apt-get install aria2 -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1 [73.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libaria2-0 amd64 1.37.0+debian-1build3 [1,105 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 aria2 amd64 1.37.0+debian-1build3 [387 kB]
Fetched 1,566 kB in 1s (1,514 kB/s)
Selecting previously unselected package libcares2:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubu

In [ ]:
https://huggingface.co/bartowski/Qwen2.5-72B-Instruct-GGUF/resolve/main/Qwen2.5-72B-Instruct-Q4_K_M.gguf

In [ ]:

aria2c -x 16 -s 16 -k 1M -c "https://huggingface.co/bartowski/Qwen2.5-72B-Instruct-GGUF/resolve/main/Qwen2.5-72B-Instruct-Q4_K_M.gguf"


In [ ]:

Qwen2.5-72B-Instruct-Q4_K_M.gguf

In [9]:
!rm -rf /content/sample_data /content/llama-b11721-bin-ubuntu-x64.tar.gz /root/.cache

In [10]:
!rm -rf /root/.cache/pip

In [11]:
!rm -rf /root/.cache/huggingface/hub/*

In [ ]:
import gc
gc.collect()

In [ ]:
Q3_K

In [17]:
%cd quant/gguf
# تشغيل السكربت على النموذج الصغير وبصيغة واحدة فقط
!bash run_quant.sh /content/Qwen2.5-72B-Instruct-Q4_K_M.gguf "Q3_K" --allow-requantize

[Errno 2] No such file or directory: 'quant/gguf'
/content/gptq-gguf-toolkit/quant/gguf
[INFO] === GGUF Model Quantization ===
[INFO] Input model: Qwen2.5-72B-Instruct-Q4_K_M (Size: 45G)
[INFO] Output directory: /content/gptq-gguf-toolkit/quant/gguf/quantized_models
[INFO] Quantization types: Q3_K

[INFO] Testing llama-quantize...
[SUCCESS] llama-quantize is working

[INFO] Using --pure option for Q3_K quantization
[INFO] Quantizing to Q3_K (3.4375 bits)
[INFO] Output: Qwen2.5-72B-Instruct-Q4_K_M_Q3_K.gguf
load_backend: loaded RPC backend from /content/llama-b11721/libggml-rpc.so
load_backend: loaded CPU backend from /content/llama-b11721/libggml-cpu-haswell.so
version: 0.7.0-dev (build 11721, commit ff5a23376)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/Qwen2.5-72B-Instruct-Q4_K_M.gguf' to '/content/gptq-gguf-toolkit/quant/gguf/quantized_models/Qwen2.5-72B-Instruct-Q4_K_M_Q3_K.gguf' as Q3_K
llama_model_loader: loaded meta data with 39 key-value pairs an

In [16]:
!rm -rf /content/gptq-gguf-toolkit/quant/gguf/quantized_models

In [ ]:
/content/gptq-gguf-toolkit/quant/gguf/run_quant.sh


#!/bin/bash

# GGUF Model Quantization Script using llama-quantize
# Usage: ./quantize_gguf.sh <model_path> [quantization_types] [--output-dir <path>] [--imatrix <path>]

set -e

# Default quantization types (بدء التكميم بصيغ IQ أولاً)
DEFAULT_QUANT_TYPES="IQ1_S IQ2_XXS IQ2_XS IQ2_S IQ2_M IQ3_XXS IQ3_S IQ3_M IQ4_NL IQ4_XS Q2_K Q3_K Q4_K Q5_K Q6_K Q8_K"

# Default paths (يفحص مسار كولاب ثم المسار المحلي)
if [ -f "/content/llama-b11721/llama-quantize" ]; then
    K_QUANT_BINARY="/content/llama-b11721/llama-quantize"
elif [ -f "$HOME/bin/llama/llama-quantize" ]; then
    K_QUANT_BINARY="$HOME/bin/llama/llama-quantize"
else
    K_QUANT_BINARY="llama-quantize"
fi

DEFAULT_OUTPUT_DIR="./quantized_models"

print_status() {
    echo -e "[INFO] $1"
}

print_success() {
    echo -e "[SUCCESS] $1"
}

print_warning() {
    echo -e "[WARNING] $1"
}

print_error() {
    echo -e "[ERROR] $1"
}

show_usage() {
    cat << EOF
Usage: $0 <model_path> [quantization_types] [options]

Arguments:
    model_path          Path to the original GGUF model file
    quantization_types  Space-separated list of quantization types (optional)
                       Default: $DEFAULT_QUANT_TYPES

Options:
    --output-dir        Directory where quantized models will be saved
                       Default: ./quantized_models
    --imatrix          Path to the imatrix file (Required for most IQ quants!)
    --allow-requantize  Allow re-quantizing an already quantized model
    --no-pure          Don't use --pure flag for K-quant quantization
    --llama-quantize   Path to llama-quantize binary
                       Default: $K_QUANT_BINARY

Available quantization types (Starting from IQ):
    IQ1_S, IQ2_XXS, IQ2_XS, IQ2_S, IQ2_M
    IQ3_XXS, IQ3_S, IQ3_M, IQ4_NL, IQ4_XS
    Q2_K, Q3_K, Q4_K, Q5_K, Q6_K, Q8_K

Examples:
    # Quantize with default types (starts from IQ)
    $0 model.gguf

    # Quantize with specific IQ types
    $0 model.gguf "IQ2_XXS IQ3_XXS IQ4_NL" --imatrix imatrix.dat

    # Custom output directory
    $0 model.gguf --output-dir ./my_quantized_models
EOF
}

validate_quant_type() {
    local quant_type="$1"
    local valid_types="IQ1_S IQ2_XXS IQ2_XS IQ2_S IQ2_M IQ3_XXS IQ3_S IQ3_M IQ4_NL IQ4_XS Q2_K Q3_K Q4_K Q5_K Q6_K Q8_K"

    for valid in $valid_types; do
        if [ "$quant_type" = "$valid" ]; then
            return 0
        fi
    done
    return 1
}

get_file_size() {
    local file="$1"
    if [ -f "$file" ]; then
        du -h "$file" | cut -f1
    else
        echo "N/A"
    fi
}

get_exact_bitwidth() {
    local quant_type="$1"
    case "$quant_type" in
        IQ1_S) echo "1.5625" ;;
        IQ2_XXS) echo "2.0625" ;;
        IQ2_XS) echo "2.3125" ;;
        IQ2_S) echo "2.5" ;;
        IQ2_M) echo "2.7" ;;
        IQ3_XXS) echo "3.0625" ;;
        IQ3_S) echo "3.4375" ;;
        IQ3_M) echo "3.66" ;;
        IQ4_NL) echo "4.25" ;;
        IQ4_XS) echo "4.25" ;;
        Q2_K) echo "2.5625" ;;
        Q3_K) echo "3.4375" ;;
        Q4_K) echo "4.5" ;;
        Q5_K) echo "5.5" ;;
        Q6_K) echo "6.5625" ;;
        Q8_K) echo "8.5" ;;
        *) echo "unknown" ;;
    esac
}

quantize_model() {
    local input_model="$1"
    local output_model="$2"
    local quant_type="$3"
    local imatrix_file="$4"
    local use_pure="$5"
    local allow_requantize="$6"

    # Trim whitespace
    quant_type=$(echo "$quant_type" | xargs)

    if [ -f "$output_model" ]; then
        print_warning "Output model already exists: $(basename "$output_model"). Skipping."
        return 0
    fi

    local quant_args=()

    # Add allow-requantize if enabled
    if [ "$allow_requantize" = true ]; then
        quant_args+=(--allow-requantize)
    fi

    # Add imatrix if provided
    if [ -n "$imatrix_file" ] && [ -f "$imatrix_file" ]; then
        print_status "Using imatrix file: $(basename "$imatrix_file")"
        quant_args+=(--imatrix "$imatrix_file")
    fi

    # Add --pure flag for certain quantization types
    if [ "$use_pure" = true ] && [ "$quant_type" != "Q8_K" ] && [[ ! "$quant_type" =~ ^IQ[12] ]]; then
        print_status "Using --pure option for $quant_type quantization"
        quant_args+=(--pure)
    fi

    quant_args+=("$input_model" "$output_model" "$quant_type")

    print_status "Quantizing to $quant_type ($(get_exact_bitwidth "$quant_type") bits)"
    print_status "Output: $(basename "$output_model")"

    # Execute quantization
    if "$K_QUANT_BINARY" "${quant_args[@]}"; then
        if [ -f "$output_model" ]; then
            local size=$(get_file_size "$output_model")
            print_success "Quantization completed: $quant_type (Size: $size)"
            return 0
        else
            print_error "Quantization completed but output file not found"
            return 1
        fi
    else
        print_error "Failed to quantize to $quant_type"
        return 1
    fi
}

main() {
    if [ $# -lt 1 ]; then
        show_usage
        exit 1
    fi

    local model_path="$1"
    local quant_types="$DEFAULT_QUANT_TYPES"
    local output_dir="$DEFAULT_OUTPUT_DIR"
    local imatrix_file=""
    local use_pure=true
    local allow_requantize=false

    # Check if model exists
    if [ ! -f "$model_path" ]; then
        print_error "Model file not found: $model_path"
        exit 1
    fi

    shift

    # Parse arguments
    while [ $# -gt 0 ]; do
        case "$1" in
            --output-dir)
                output_dir="$2"
                shift 2
                ;;
            --imatrix)
                imatrix_file="$2"
                shift 2
                ;;
            --allow-requantize)
                allow_requantize=true
                shift
                ;;
            --no-pure)
                use_pure=false
                shift
                ;;
            --llama-quantize)
                K_QUANT_BINARY="$2"
                shift 2
                ;;
            --help|-h)
                show_usage
                exit 0
                ;;
            -*)
                print_error "Unknown option: $1"
                show_usage
                exit 1
                ;;
            *)
                # Assume it's quantization types
                quant_types="$1"
                shift
                ;;
        esac
    done

    # Validate llama-quantize binary
    if ! command -v "$K_QUANT_BINARY" &> /dev/null && [ ! -x "$K_QUANT_BINARY" ]; then
        print_error "llama-quantize not found at: $K_QUANT_BINARY"
        print_error "Please install llama.cpp or specify path with --llama-quantize"
        exit 1
    fi

    # Validate imatrix file if provided
    if [ -n "$imatrix_file" ] && [ ! -f "$imatrix_file" ]; then
        print_error "imatrix file not found: $imatrix_file"
        exit 1
    fi

    # Create output directory
    mkdir -p "$output_dir"

    # Get absolute paths
    model_path=$(realpath "$model_path")
    output_dir=$(realpath "$output_dir")
    [ -n "$imatrix_file" ] && imatrix_file=$(realpath "$imatrix_file")

    local model_name=$(basename "$model_path" .gguf)
    local model_size=$(get_file_size "$model_path")

    # Print header
    print_status "=== GGUF Model Quantization ==="
    print_status "Input model: $model_name (Size: $model_size)"
    print_status "Output directory: $output_dir"
    print_status "Quantization types: $quant_types"
    [ -n "$imatrix_file" ] && print_status "imatrix: $(basename "$imatrix_file")"
    echo

    # Test llama-quantize
    print_status "Testing llama-quantize..."
    if "$K_QUANT_BINARY" --help 2>&1 | grep -q -i "usage"; then
        print_success "llama-quantize is working"
    else
        print_warning "Could not verify llama-quantize. Continuing anyway..."
    fi
    echo

    # Process each quantization type
    local success_count=0
    local total_count=0
    local successful_quants=()

    for quant_type in $quant_types; do
        if ! validate_quant_type "$quant_type"; then
            print_warning "Invalid quantization type: $quant_type (skipping)"
            continue
        fi

        total_count=$((total_count + 1))

        local output_file="$output_dir/${model_name}_${quant_type}.gguf"

        if quantize_model "$model_path" "$output_file" "$quant_type" "$imatrix_file" "$use_pure" "$allow_requantize"; then
            success_count=$((success_count + 1))
            successful_quants+=("$quant_type")
        fi
        echo
    done

    # Create summary JSON
    local summary_file="$output_dir/quantization_summary.json"
    cat > "$summary_file" << EOF
{
  "original_model": "$model_name",
  "original_size": "$model_size",
  "timestamp": "$(date -u +"%Y-%m-%dT%H:%M:%SZ")",
  "quantizations": [
EOF

    local first=true
    for quant in "${successful_quants[@]}"; do
        [ "$first" = false ] && echo "," >> "$summary_file"
        first=false
        local output_file="$output_dir/${model_name}_${quant}.gguf"
        local size=$(get_file_size "$output_file")
        cat >> "$summary_file" << EOF
    {
      "type": "$quant",
      "bitwidth": "$(get_exact_bitwidth "$quant")",
      "file": "$(basename "$output_file")",
      "size": "$size"
    }
EOF
    done

    cat >> "$summary_file" << EOF

  ],
  "imatrix_used": $([ -n "$imatrix_file" ] && echo "true" || echo "false")
}
EOF

    # Print summary
    echo
    print_status "=== QUANTIZATION SUMMARY ==="
    print_status "Successful: $success_count/$total_count"
    print_status "Output directory: $output_dir"

    if [ ${#successful_quants[@]} -gt 0 ]; then
        echo
        print_status "Quantized models:"
        for quant in "${successful_quants[@]}"; do
            local output_file="$output_dir/${model_name}_${quant}.gguf"
            print_status "  • $quant ($(get_exact_bitwidth "$quant") bits): $(get_file_size "$output_file")"
        done
    fi

    if [ $success_count -gt 0 ]; then
        print_success "Quantization completed successfully!"
        print_status "Summary saved to: $summary_file"
    else
        print_error "No quantizations were successful!"
        exit 1
    fi
}

# Run main function
if [ "${BASH_SOURCE[0]}" = "${0}" ]; then
    main "$@"
fi

In [18]:

from huggingface_hub import login

# ضع التوكن الخاص بك هنا
HF_TOKEN = "XXXXXXXXXXXXXXX"
login(token=HF_TOKEN)




In [19]:






from huggingface_hub import HfApi, login

# 1. تسجيل الدخول (استبدل hf_xxx بالتوكن الخاص بك بصلاحية Write)
login(token="XXXXXXXXXXXXXXX")

# 2. تحديد المعرف والمسار
repo_id = "zed122/gptq-gguf-toolkit"
local_file = "/content/gptq-gguf-toolkit/quant/gguf/quantized_models/Qwen2.5-72B-Instruct-Q4_K_M_Q3_K.gguf"
path_in_repo = "Qwen2.5-72B-Instruct-Q4_K_M_Q3_K.gguf"  # الاسم الذي سيظهر به الملف داخل الريبو

# 3. رفع الملف
api = HfApi()

api.upload_file(
    path_or_fileobj=local_file,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model"
)

print("تم رفع الملف بنجاح!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...Instruct-Q4_K_M_Q3_K.gguf:   0%|          |  185kB / 33.8GB            

Validating                    :           |   0%            

تم رفع الملف بنجاح!


In [20]:
!rm -rf /content/gptq-gguf-toolkit/quant/gguf/quantized_models

In [2]:
%cd /content/gptq-gguf-toolkit/quant/gguf
# تشغيل السكربت على النموذج الصغير وبصيغة واحدة فقط
!bash run_quant.sh /content/Qwen2.5-72B-Instruct-Q4_K_M.gguf "Q2_K" --allow-requantize

/content/gptq-gguf-toolkit/quant/gguf
[INFO] === GGUF Model Quantization ===
[INFO] Input model: Qwen2.5-72B-Instruct-Q4_K_M (Size: 45G)
[INFO] Output directory: /content/gptq-gguf-toolkit/quant/gguf/quantized_models
[INFO] Quantization types: Q2_K

[INFO] Testing llama-quantize...
[SUCCESS] llama-quantize is working

[INFO] Using --pure option for Q2_K quantization
[INFO] Quantizing to Q2_K (2.5625 bits)
[INFO] Output: Qwen2.5-72B-Instruct-Q4_K_M_Q2_K.gguf
load_backend: loaded RPC backend from /content/llama-b11721/libggml-rpc.so
load_backend: loaded CPU backend from /content/llama-b11721/libggml-cpu-haswell.so
version: 0.7.0-dev (build 11721, commit ff5a23376)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/Qwen2.5-72B-Instruct-Q4_K_M.gguf' to '/content/gptq-gguf-toolkit/quant/gguf/quantized_models/Qwen2.5-72B-Instruct-Q4_K_M_Q2_K.gguf' as Q2_K
llama_model_loader: loaded meta data with 39 key-value pairs and 963 tensors from /content/Qwen2.5-72B-Instruct-Q

https://huggingface.co/bartowski/Qwen2.5-72B-Instruct-GGUF

https://huggingface.co/zed122/gptq-gguf-toolkit

https://github.com/IST-DASLab/gptq-gguf-toolkit/tree/main

In [3]:

from huggingface_hub import login

# ضع التوكن الخاص بك هنا
HF_TOKEN = "XXXXXXXXXXXXXXX"
login(token=HF_TOKEN)




In [4]:






from huggingface_hub import HfApi, login

# 1. تسجيل الدخول (استبدل hf_xxx بالتوكن الخاص بك بصلاحية Write)
login(token="XXXXXXXXXXXXXXX")

# 2. تحديد المعرف والمسار
repo_id = "zed122/gptq-gguf-toolkit"
local_file = "/content/gptq-gguf-toolkit/quant/gguf/quantized_models/Qwen2.5-72B-Instruct-Q4_K_M_Q2_K.gguf"
path_in_repo = "Qwen2.5-72B-Instruct-Q4_K_M_Q2_K.gguf"  # الاسم الذي سيظهر به الملف داخل الريبو

# 3. رفع الملف
api = HfApi()

api.upload_file(
    path_or_fileobj=local_file,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model"
)

print("تم رفع الملف بنجاح!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...Instruct-Q4_K_M_Q2_K.gguf:   0%|          | 5.91MB / 28.4GB            

Validating                    :           |   0%            

تم رفع الملف بنجاح!


In [ ]:
!/content/llama-b11721/llama-cli \
  -m /content/gptq-gguf-toolkit/quant/gguf/quantized_models/Qwen2.5-72B-Instruct-Q4_K_M_Q2_K.gguf \
  -lm mmap \
  -lzm on \
  -rea off \
  --reasoning-budget 0 \
  -ngl 0 \
  -ctk q4_0 \
  -ctv q4_0 \
  -c 512 \
  -n 44



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\

https://huggingface.co/ISTA-DASLab/Llama-3.1-8B-Instruct-GGUF